In [1]:
# ─────────────────────────────────────────────
# Cell 1 | Import libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import shap
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
print("Libraries loaded")


# ─────────────────────────────────────────────
# Cell 2 | Load preprocessed data
# ─────────────────────────────────────────────
# Load the full preprocessed dataset (includes dsr column)
df = pd.read_csv('../data/youth_debt_preprocessed.csv')

print(f"Shape : {df.shape}")
print(f"DSR range : {df['dsr'].min():.3f} ~ {df['dsr'].max():.3f}")
print(f"DSR missing : {df['dsr'].isna().sum()}")


# ─────────────────────────────────────────────
# Cell 3 | Define feature columns
# (same as 03_model_lgbm_xgb.ipynb)
# ─────────────────────────────────────────────
DROP_COLS = [
    'h19_pid', 'p19_wgc', 'h19_g4',
    'h19_pers_income3', 'h19_pers_income5',
    'h1906_10', 'dsr_raw', 'dsr',
    'annual_debt_service', 'annual_interest',
    'annual_repayment', 'total_income',
    'h1909_aq7', 'h1909_aq8', 'h1906_aq2',
    'debt_crisis',  # target — exclude from features
]

FEAT_COLS = [c for c in df.columns if c not in DROP_COLS]
print(f"Feature cols : {len(FEAT_COLS)}")
print(FEAT_COLS)


# ─────────────────────────────────────────────
# Cell 4 | Run sensitivity analysis
# ─────────────────────────────────────────────
results = []

for thr in [0.30, 0.40, 0.50]:

    # Redefine target at this threshold
    df_thr = df.dropna(subset=['dsr']).copy()
    df_thr['debt_crisis'] = (df_thr['dsr'] >= thr).astype(int)

    X = df_thr[FEAT_COLS]
    y = df_thr['debt_crisis']

    crisis_rate = y.mean()

    # Impute
    imp = SimpleImputer(strategy='median')
    X_imp = pd.DataFrame(imp.fit_transform(X), columns=FEAT_COLS)

    # Train/test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_imp, y, test_size=0.2, random_state=42, stratify=y)

    # SMOTE on training set
    sm = SMOTE(random_state=42)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=500, learning_rate=0.05,
        max_depth=6, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42, verbose=-1
    )
    lgbm.fit(X_tr_sm, y_tr_sm)

    # 5-fold CV AUC
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(lgbm, X_imp, y,
                                cv=cv, scoring='roc_auc')
    cv_auc = cv_scores.mean()

    # Test AUC
    y_prob   = lgbm.predict_proba(X_te)[:, 1]
    test_auc = roc_auc_score(y_te, y_prob)

    # SHAP top 3
    explainer = shap.TreeExplainer(lgbm)
    sv = explainer.shap_values(X_te)
    if isinstance(sv, list):
        sv = sv[1]
    mean_shap = pd.Series(
        np.abs(sv).mean(axis=0), index=FEAT_COLS
    ).sort_values(ascending=False)
    top3 = mean_shap.index[:3].tolist()

    results.append({
        'threshold'   : thr,
        'crisis_rate' : f"{crisis_rate:.1%}",
        'cv_auc'      : round(cv_auc, 3),
        'test_auc'    : round(test_auc, 3),
        'top3_shap'   : ', '.join(top3),
    })

    print(f"DSR>={thr:.2f} | crisis={crisis_rate:.1%} "
          f"| CV AUC={cv_auc:.3f} | Test AUC={test_auc:.3f} "
          f"| Top3: {top3}")


# ─────────────────────────────────────────────
# Cell 5 | Display & save results
# ─────────────────────────────────────────────
results_df = pd.DataFrame(results)
print("\n=== DSR Threshold Sensitivity Results ===")
print(results_df.to_string(index=False))

results_df.to_csv('../outputs/sensitivity_dsr_threshold.csv',
                  index=False, encoding='utf-8-sig')
print("\nSaved -> outputs/sensitivity_dsr_threshold.csv")

Libraries loaded
Shape : (1916, 37)
DSR range : 0.000 ~ 5.000
DSR missing : 0
Feature cols : 21
['sex', 'education', 'marital_status', 'region', 'employment_status', 'employment_type', 'h19_pers_income1', 'h19_pers_income2', 'h19_pers_income4', 'h19_cin', 'h19_din', 'debt_financial', 'debt_private', 'debt_card', 'debt_lease', 'debt_credit', 'debt_other', 'housing_debt_balance', 'housing_tenure', 'age', 'total_debt']
DSR>=0.30 | crisis=18.8% | CV AUC=0.896 | Test AUC=0.875 | Top3: ['debt_financial', 'h19_pers_income2', 'housing_debt_balance']
DSR>=0.40 | crisis=15.6% | CV AUC=0.884 | Test AUC=0.850 | Top3: ['debt_financial', 'h19_pers_income2', 'sex']
DSR>=0.50 | crisis=13.1% | CV AUC=0.882 | Test AUC=0.881 | Top3: ['h19_pers_income2', 'debt_financial', 'housing_tenure']

=== DSR Threshold Sensitivity Results ===
 threshold crisis_rate  cv_auc  test_auc                                              top3_shap
       0.3       18.8%   0.896     0.875 debt_financial, h19_pers_income2, housi